# AI in Industry — Lab

**Nothing installs on your computer.** Everything runs on Google's servers; your laptop only needs a browser.

Do the three steps below, then run the **Setup** cell.

---

## Step 1 — Make your own copy, then close the old tab

Click **File → Save a copy in Drive**.

A new tab opens, titled **`Copy of lab.ipynb`**.

> **Now close the tab you were in before.** You have two tabs open and they look almost identical. The old one is not yours — nothing you type there is saved, and it is easy to spend the whole session in the wrong tab.
>
> Before going on, check the title at the top left says **`Copy of lab.ipynb`**. If it does not, you are in the wrong tab.

---

## Step 2 — Get a free API key

Open **[console.groq.com](https://console.groq.com/keys)** in a new tab → sign in with Google → **API Keys** → **Create API Key** → name it `lab` → **Submit**.

Copy the key. It starts with `gsk_` and is **shown only once** — if you lose it, delete that key and make another. It is free.

---

## Step 3 — Store the key in Colab

Do **not** paste your key into a code cell. Colab has a proper place for it.

1. On the **far left edge**, click the **🔑 key icon**. The **Secrets** panel opens.
2. Click **+ Add new secret**.
3. A row appears with four columns. Fill in the middle two, then switch on the first:

| Column | What to do |
|---|---|
| **Name** | Type `LLM_API_KEY` — all capitals, two underscores |
| **Value** | Paste your `gsk_...` key |
| **Notebook access** | **Click the toggle so it turns on** |
| Actions | Leave alone |

### The two things that go wrong here

**The Notebook access toggle starts OFF.** It shows a grey ✕ until you click it. If you leave it off, the notebook cannot see your key and Setup fails at stage 4 — even though the secret looks saved.

**The Name box is narrow and hides the end of what you typed.** It may display `LLM_API` when you have typed the whole thing, or when you have not. Click into the box and press `End` to see the full text. It must read `LLM_API_KEY` exactly — a name that is close but not identical will not be found.

---

Now run the **Setup** cell below. It takes 2–3 minutes the first time.

## Setup

Run this once. It takes **2–3 minutes the first time** — it is downloading the lab files and a small language model. Let it finish.

In [ ]:
#@title Setup - run me first { display-mode: "form" }
# Verbose on purpose: every step says what it is doing and what to do if it fails.
import os, sys, pathlib, subprocess, time

REPO, NAME, SUB = "coolMukul/rough", "rough", "ai-lab"
BASE = "/content" if pathlib.Path("/content").exists() else "."
t0 = time.time()

def step(n, msg):  print(f"\n[{n}/5] {msg}", flush=True)
def ok(msg):       print(f"      OK - {msg}", flush=True)
def die(msg, fix):
    print(f"\n      FAILED - {msg}\n")
    for line in fix.strip().splitlines():
        print("      " + line.strip())
    print("\n      Ask for help, then move on. Each later step sets itself up,")
    print("      so a failure here does not stop you joining in later.\n")
    raise SystemExit(1)

def run(cmd):
    """Run a shell command, capture everything, return (code, output)."""
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return p.returncode, (p.stdout + p.stderr).strip()

def secret(name):
    try:
        from google.colab import userdata
        return (userdata.get(name) or "").strip()
    except Exception as exc:
        print(f"      (could not read secret {name}: {type(exc).__name__})")
        return ""

print("=" * 64)
print("  AI in Industry - lab setup")
print("=" * 64)

# ---------------------------------------------------------------- 1. environment
step(1, "Checking the environment")
print(f"      python {sys.version.split()[0]}")
print(f"      running in {'Google Colab' if pathlib.Path('/content').exists() else 'local'}")
code, out = run("git --version")
ok(out if code == 0 else "git missing")

# ---------------------------------------------------------------- 2. lab files
step(2, "Downloading the lab files")
os.chdir(BASE)
if pathlib.Path(NAME).exists():
    # Already here from an earlier run - refresh rather than reuse stale files.
    print("      already present, checking for updates")
    code, out = run(f"cd {NAME} && git pull --quiet --ff-only")
    ok("up to date" if code == 0 else "could not update, using existing copy")
else:
    print(f"      cloning {REPO} (public)")
    code, out = run(f"git clone --depth 1 https://github.com/{REPO}.git {NAME}")
    if code != 0:
        die(out, """
            Check your internet connection and run this cell again.
            If it keeps failing, tell the lecturer - the repository
            address may have changed.
            """)
    ok("downloaded")

os.chdir(f"{BASE}/{NAME}/{SUB}")
print(f"      working directory: {os.getcwd()}")
missing = [f for f in ("requirements.txt", "verify_setup.py", "data/chunks.json") 
           if not pathlib.Path(f).exists()]
if missing:
    die("these files are missing: " + ", ".join(missing),
        "Delete the folder and run this cell again:  !rm -rf " + NAME)
ok("all expected files present")

# ---------------------------------------------------------------- 3. packages
step(3, "Installing packages - SLOW the first time (2-3 min), please wait")
code, out = run(f"{sys.executable} -m pip install -q -r requirements.txt")
if code != 0:
    print(out[-1500:])
    die("pip install failed", "Runtime -> Restart session, then run this cell again.")
ok("installed")

# ---------------------------------------------------------------- 4. API key
step(4, "Looking for your API key")
key = secret("LLM_API_KEY")
if not key:
    die("no key found in Colab secrets", """
        1. Click the key icon on the far left edge
        2. Click '+ Add new secret'
        3. Name it exactly:  LLM_API_KEY
           The name box is narrow and hides the end of long text.
           Click into it and press End to check the whole name is there.
        4. Paste your key from  https://console.groq.com/keys
        5. Turn ON the 'Notebook access' toggle - it starts OFF and
           shows a grey X. This is the most common mistake.
        6. Run this cell again
        """)
if key != key.strip() or len(key) < 20:
    die("the key looks malformed (too short, or has spaces around it)",
        "Re-copy it from the Groq console and update the secret.")
os.environ["LLM_API_KEY"] = key
ok(f"found, ending ...{key[-4:]}")

# ---------------------------------------------------------------- 5. verify
step(5, "Running the full check")
print()
code, out = run(f"{sys.executable} verify_setup.py")
print(out)

print("\n" + "=" * 64)
if code == 0:
    print(f"  READY. Took {time.time() - t0:.0f} seconds.")
else:
    print("  Some checks failed - read the FAIL lines above for the fix.")
    print("  If you cannot fix it now, carry on anyway: each later step")
    print("  sets itself up from scratch, so you will not be left behind.")
print("=" * 64)


### What you should see

Seven `PASS` lines, ending with `All checks passed.`

If any line says `FAIL`, the message underneath tells you the fix. The two most common:

| Message | Fix |
|---|---|
| `no API key found` | Secret must be named exactly `LLM_API_KEY`, and **Notebook access** must be on. Re-run this cell after fixing. |
| `401 unauthorised` | The key is wrong or has a space at either end. Re-copy it from the Groq console. |

**Stuck? Do not stop and debug during the session.** Skip to the next step's script and carry on — each one sets itself up from scratch, so you will be caught up automatically.

---

## During the session

Each part of the talk has its own script. You will **paste or type it into a new cell below and run it.**

Every script rebuilds everything it needs from scratch, so:

- If one step fails for you, **skip it and run the next one.** You will not be left behind.
- You can join at any point.
- Add a new cell with the **+ Code** button at the top left.

In [ ]:
# Scratch cell — paste each step's script here and run it.